# LeetCode 2335 — Minimum Amount of Time to Fill Cups bằng Python

Link đề: https://leetcode.com/problems/minimum-amount-of-time-to-fill-cups/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting, Heap

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def fillCups(self, amount: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài **tối ưu hóa thời gian** với luật "mỗi giây rót tối đa 2 cốc **khác loại**", và cách nghĩ bằng **cận dưới** (lower bound) cùng **cách xây dựng đạt cận** để chứng minh đáp án.
2. Biết dùng `max`, `sum`, `sorted`, phép **chia lấy trần** `(x + 1) // 2` hoặc `-(-x // 2)`, và hàng đợi ưu tiên `heapq` (max-heap bằng cách đảo dấu).
3. Hiểu chiến lược **tham lam (greedy)**: mỗi giây luôn rót vào **hai loại còn nhiều nhất**.
4. Viết 5 cách: quy hoạch động / đệ quy có nhớ (cho input nhỏ), mô phỏng tham lam bằng sắp xếp, mô phỏng bằng heap, công thức `max(max, ceil(sum/2))`, và công thức theo trường hợp sau khi sắp xếp.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: chỉ lấy `ceil(sum/2)` mà quên cận `max`, chia nguyên làm tròn xuống, rót vào hai loại **bất kỳ** thay vì lớn nhất.

## 2. Đề bài & ví dụ

Một máy lọc nước có thể rót **nước lạnh, nước ấm, nước nóng**. Mỗi giây, bạn có thể thực hiện **một** trong hai việc:

- rót đầy **2 cốc** có **loại nước khác nhau** (mỗi loại một cốc);
- hoặc rót đầy **1 cốc** của một loại bất kỳ.

Cho mảng `amount` độ dài `3` trong đó `amount[0]`, `amount[1]`, `amount[2]` là số cốc nước **lạnh, ấm, nóng** cần rót. Trả về **số giây tối thiểu** để rót đủ tất cả các cốc.

| Ví dụ | amount | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,4,2]` | `4` | giây 1: lạnh+ấm; giây 2: ấm+nóng; giây 3: ấm+nóng; giây 4: ấm |
| 2 | `[5,4,4]` | `7` | rót hai loại nhiều nhất mỗi giây, tổng 13 cốc, tối thiểu `ceil(13/2) = 7` |
| 3 | `[5,0,0]` | `5` | chỉ có một loại, mỗi giây một cốc |

Constraints: `amount.length == 3`; `0 <= amount[i] <= 100`.

In [1]:
examples = [([1, 4, 2], 4), ([5, 4, 4], 7), ([5, 0, 0], 5)]
for amount, expected in examples:
    print(f"amount={amount}: tổng={sum(amount)}, max={max(amount)}, ceil(tổng/2)={(sum(amount) + 1) // 2} -> mong đợi {expected}")

amount=[1, 4, 2]: tổng=7, max=4, ceil(tổng/2)=4 -> mong đợi 4
amount=[5, 4, 4]: tổng=13, max=5, ceil(tổng/2)=7 -> mong đợi 7
amount=[5, 0, 0]: tổng=5, max=5, ceil(tổng/2)=3 -> mong đợi 5


## 3. Kiến thức nền

### 3.1. `max`, `sum` và `sorted` trên list

`max(a)` lấy phần tử lớn nhất, `sum(a)` tổng, `sorted(a)` trả list mới sắp **tăng dần** (không sửa `a`). `a.sort()` sắp xếp tại chỗ. Với list 3 phần tử, mọi thứ chạy tức thì.

In [2]:
a = [5, 4, 4]
print(max(a), sum(a), sorted(a), a)
a.sort(reverse=True)
print(a)

5 13 [4, 4, 5] [5, 4, 4]
[5, 4, 4]


### 3.2. Chia lấy trần (ceiling division)

`ceil(x / 2)` là số nguyên nhỏ nhất `>= x / 2`. Dùng số nguyên: `(x + 1) // 2`, hoặc `-(-x // 2)`. Nhớ `x // 2` là chia **lấy sàn** (làm tròn xuống) — sai khi `x` lẻ.

In [3]:
import math

for x in (0, 1, 2, 3, 13):
    print(x, x // 2, (x + 1) // 2, -(-x // 2), math.ceil(x / 2))

0 0 0 0 0
1 0 1 1 1
2 1 1 1 1
3 1 2 2 2
13 6 7 7 7


### 3.3. Hàng đợi ưu tiên `heapq` (max-heap bằng đảo dấu)

`heapq` trong Python là **min-heap**: `heappop` lấy phần tử **nhỏ nhất**. Muốn lấy lớn nhất, đẩy `-x` vào và đảo dấu khi lấy ra. `heappush(h, v)` thêm, `heappop(h)` lấy `O(log n)`.

In [4]:
import heapq

h = []
for x in (5, 4, 4):
    heapq.heappush(h, -x)           # đảo dấu để đỉnh heap là số lớn nhất
print([-x for x in sorted(h)])
big = -heapq.heappop(h)
print("lấy lớn nhất:", big, "còn", sorted(-x for x in h))

[5, 4, 4]
lấy lớn nhất: 5 còn [4, 4]


### 3.4. Cận dưới (lower bound) và cách nghĩ "đạt cận"

Một kỹ thuật chứng minh đáp án tối ưu: (1) tìm **cận dưới** — con số mà **không cách nào** làm nhanh hơn; (2) chỉ ra **một cách làm đạt đúng cận đó**. Khi hai cái trùng nhau thì đó chính là đáp án. Bài này dùng đúng kỹ thuật đó.

In [5]:
amount = [1, 4, 2]
lb1 = max(amount)                 # mỗi giây một loại giảm nhiều nhất 1 cốc
lb2 = (sum(amount) + 1) // 2      # mỗi giây tổng giảm nhiều nhất 2 cốc
print("cận dưới 1:", lb1, "| cận dưới 2:", lb2, "| cận lớn hơn:", max(lb1, lb2))

cận dưới 1: 4 | cận dưới 2: 4 | cận lớn hơn: 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hai điều **chắc chắn** đúng với mọi cách rót:

1. **Mỗi loại**: mỗi giây ta chỉ rót tối đa **1 cốc** mỗi loại, nên loại có `max(amount)` cốc cần **ít nhất** `max(amount)` giây.
2. **Tổng**: mỗi giây rót tối đa **2 cốc**, nên tổng `sum(amount)` cốc cần **ít nhất** `ceil(sum / 2)` giây.

Vậy đáp án `>= max(max(amount), ceil(sum/2))`.

Tính tay `[1,4,2]`: `max = 4`, `sum = 7 -> ceil(3.5) = 4`; cận lớn nhất là `4`; đề cho đáp án `4` nên đạt cận. Với `[5,4,4]`: `max = 5`, `sum = 13 -> 7`; đáp án `7` (cận thứ hai). Với `[5,0,0]`: `max = 5`, `sum = 5 -> 3`; đáp án `5` (cận thứ nhất).

Câu hỏi còn lại: **có luôn đạt được cận này không?** Có — bằng chiến lược rót hai loại còn nhiều nhất (xem Mục 6 và Mục 10).

In [6]:
for amount in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [2, 2, 2]):
    a, b = max(amount), (sum(amount) + 1) // 2
    print(f"{amount}: max={a}, ceil(sum/2)={b} -> cận dưới={max(a, b)}")

[1, 4, 2]: max=4, ceil(sum/2)=4 -> cận dưới=4
[5, 4, 4]: max=5, ceil(sum/2)=7 -> cận dưới=7
[5, 0, 0]: max=5, ceil(sum/2)=3 -> cận dưới=5
[2, 2, 2]: max=2, ceil(sum/2)=3 -> cận dưới=3


## 5. Cách 1 — Brute Force / baseline: quy hoạch động / đệ quy có nhớ

**Ý tưởng:** thử **mọi** cách rót. Trạng thái là bộ ba còn lại `(a, b, c)`. Mỗi giây chọn: rót 1 cốc loại `i` (nếu còn), hoặc rót 2 cốc loại `i`, `j` khác nhau (nếu cả hai còn). Đáp án `f(a,b,c) = 1 + min(f(trạng thái kế))`, `f(0,0,0) = 0`. Dùng `functools.lru_cache` để không tính lại.

**Độ phức tạp:** có `O(a*b*c)` trạng thái, mỗi trạng thái `O(1)` chuyển (6 lựa chọn): với `100^3 = 10^6` trạng thái là hơi nặng và đệ quy sâu quá giới hạn mặc định, nên chỉ dùng để **kiểm tra** trên input nhỏ. **Nút thắt:** không cần thử mọi cách, tham lam đạt cận (Mục 6).

In [7]:
from typing import List
from functools import lru_cache

class SolutionDP:
    def fillCups(self, amount: List[int]) -> int:
        @lru_cache(maxsize=None)
        def f(a, b, c):
            if a == b == c == 0:
                return 0
            best = float("inf")
            state = [a, b, c]
            for i in range(3):                       # rót 1 cốc loại i
                if state[i] > 0:
                    nxt = state[:]
                    nxt[i] -= 1
                    best = min(best, 1 + f(*sorted(nxt)))
            for i in range(3):                       # rót 2 cốc hai loại khác nhau i < j
                for j in range(i + 1, 3):
                    if state[i] > 0 and state[j] > 0:
                        nxt = state[:]
                        nxt[i] -= 1
                        nxt[j] -= 1
                        best = min(best, 1 + f(*sorted(nxt)))
            return best
        return f(*sorted(amount))

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [0, 0, 0]):
    print(amt, "->", SolutionDP().fillCups(amt))

[1, 4, 2] -> 4
[5, 4, 4] -> 7
[5, 0, 0] -> 5
[0, 0, 0] -> 0


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** mỗi giây hãy rót vào **hai loại có nhiều cốc nhất** (nếu có ít nhất hai loại còn cốc), nếu không thì rót nốt loại còn lại. Lý do: ta muốn tránh tình huống một loại còn quá nhiều trong khi hai loại kia đã hết — lúc đó chỉ rót được 1 cốc/giây, lãng phí.

**Invariant:** sau mỗi bước tham lam, hiệu số lớn nhất không bị "bỏ lại": nếu ban đầu `max <= ceil(sum/2)` thì mãi mãi giữ `max <= ceil(sum_còn_lại / 2)` — vì ta luôn giảm loại lớn nhất.

**Kết luận:** đáp án đạt đúng cận dưới: `max(max(amount), ceil(sum(amount) / 2))`.

Hai trường hợp (khi sắp xếp `x <= y <= z`):

- **`z >= x + y`:** loại nhiều nhất `z` lớn đến mức dù ghép với `x`, `y` mỗi giây vẫn còn thừa; đáp án `z` (cứ ghép `z` với một cốc của `x` hoặc `y` mỗi giây, phần thừa rót một mình).
- **`z < x + y`:** cứ ghép sao cho không lãng phí, mỗi giây rót đủ 2 cốc, ngoại trừ có thể 1 giây cuối chỉ 1 cốc nếu tổng lẻ: đáp án `ceil((x + y + z) / 2)`.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng tham lam bằng sắp xếp mỗi vòng

Mỗi giây: sắp xếp giảm dần; trừ `1` cho hai phần tử đầu (nếu `> 0`); tăng bộ đếm giây; lặp cho tới khi `amount[0] == 0` (phần tử lớn nhất bằng 0).

In [8]:
from typing import List

class SolutionSortLoop:
    def fillCups(self, amount: List[int]) -> int:
        a = sorted(amount, reverse=True)
        seconds = 0
        while a[0] > 0:
            a[0] -= 1
            if a[1] > 0:
                a[1] -= 1
            a.sort(reverse=True)
            seconds += 1
        return seconds

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [0, 0, 0]):
    print(amt, "->", SolutionSortLoop().fillCups(amt))

[1, 4, 2] -> 4
[5, 4, 4] -> 7
[5, 0, 0] -> 5
[0, 0, 0] -> 0


**Độ phức tạp Cách 2:** số giây tối đa `~ sum/2 ... max` nên tối đa 300 vòng với `amount[i] <= 100`; mỗi vòng sắp xếp 3 phần tử `O(1)`. Tổng `O(S)` với `S = sum(amount)` (hoặc `O(max)`), Space `O(1)`.

### Cách 3 — Mô phỏng bằng max-heap

Dùng `heapq` với giá trị đảo dấu. Mỗi giây lấy hai phần tử lớn nhất, giảm `1`, đẩy lại những phần tử còn `> 0`.

In [9]:
import heapq
from typing import List

class SolutionHeap:
    def fillCups(self, amount: List[int]) -> int:
        heap = [-x for x in amount if x > 0]
        heapq.heapify(heap)
        seconds = 0
        while heap:
            first = -heapq.heappop(heap)
            second = -heapq.heappop(heap) if heap else 0
            first -= 1
            second -= 1
            if first > 0:
                heapq.heappush(heap, -first)
            if second > 0:
                heapq.heappush(heap, -second)
            seconds += 1
        return seconds

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [0, 0, 0]):
    print(amt, "->", SolutionHeap().fillCups(amt))

[1, 4, 2] -> 4
[5, 4, 4] -> 7
[5, 0, 0] -> 5
[0, 0, 0] -> 0


**Độ phức tạp Cách 3:** `O(S log 3) = O(S)` vì heap tối đa 3 phần tử, Space `O(1)`. Cách này mở rộng tốt khi có **nhiều loại nước** hơn 3.

### Cách 4 — Công thức `max(max(amount), ceil(sum / 2))`

Hai cận dưới đạt được nên đáp án là lớn hơn trong hai cận. Một dòng, `O(1)`.

In [10]:
from typing import List

class SolutionFormula:
    def fillCups(self, amount: List[int]) -> int:
        return max(max(amount), (sum(amount) + 1) // 2)

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [0, 0, 0]):
    print(amt, "->", SolutionFormula().fillCups(amt))

[1, 4, 2] -> 4
[5, 4, 4] -> 7
[5, 0, 0] -> 5
[0, 0, 0] -> 0


**Độ phức tạp Cách 4:** Time `O(1)` (mảng độ dài 3), Space `O(1)`.

### Cách 5 — Sắp xếp rồi chia hai trường hợp `z >= x + y`

Sắp xếp `x <= y <= z`. Nếu `z >= x + y` thì đáp án `z`. Ngược lại `ceil((x+y+z)/2)`. Cách này làm lộ **vì sao** công thức đúng, hữu ích khi giải thích.

In [11]:
from typing import List

class SolutionCases:
    def fillCups(self, amount: List[int]) -> int:
        x, y, z = sorted(amount)
        if z >= x + y:
            return z                       # loại nhiều nhất quá lớn: bị giới hạn bởi chính nó
        return (x + y + z + 1) // 2       # đủ cân bằng: mỗi giây rót đủ 2 cốc (trừ có thể giây cuối)

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [3, 3, 3]):
    print(amt, "->", SolutionCases().fillCups(amt))

[1, 4, 2] -> 4
[5, 4, 4] -> 7
[5, 0, 0] -> 5
[3, 3, 3] -> 5


**Độ phức tạp Cách 5:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 (tham lam) với `amount = [5,4,4]`:

| Giây | Trước (đã sắp giảm) | Rót vào | Sau | Tổng còn |
|---|---|---|---|---|
| 1 | `[5,4,4]` | loại 5 và 4 | `[4,3,4] -> [4,4,3]` | 11 |
| 2 | `[4,4,3]` | 4 và 4 | `[3,3,3]` | 9 |
| 3 | `[3,3,3]` | 3 và 3 | `[2,2,3] -> [3,2,2]` | 7 |
| 4 | `[3,2,2]` | 3 và 2 | `[2,1,2] -> [2,2,1]` | 5 |
| 5 | `[2,2,1]` | 2 và 2 | `[1,1,1]` | 3 |
| 6 | `[1,1,1]` | 1 và 1 | `[0,0,1] -> [1,0,0]` | 1 |
| 7 | `[1,0,0]` | chỉ 1 loại | `[0,0,0]` | 0 |

Tổng 7 giây `= ceil(13/2) = 7`.

In [12]:
def trace_greedy(amount):
    a = sorted(amount, reverse=True)
    seconds = 0
    print(f"amount={amount}")
    while a[0] > 0:
        before = a[:]
        a[0] -= 1
        used = 1
        if a[1] > 0:
            a[1] -= 1
            used = 2
        a.sort(reverse=True)
        seconds += 1
        print(f"  giây {seconds}: {before} rót {used} cốc -> {a} (tổng còn {sum(a)})")
    return seconds

print("kết quả:", trace_greedy([5, 4, 4]))
print("kết quả:", trace_greedy([1, 4, 2]))

amount=[5, 4, 4]
  giây 1: [5, 4, 4] rót 2 cốc -> [4, 4, 3] (tổng còn 11)
  giây 2: [4, 4, 3] rót 2 cốc -> [3, 3, 3] (tổng còn 9)
  giây 3: [3, 3, 3] rót 2 cốc -> [3, 2, 2] (tổng còn 7)
  giây 4: [3, 2, 2] rót 2 cốc -> [2, 2, 1] (tổng còn 5)
  giây 5: [2, 2, 1] rót 2 cốc -> [1, 1, 1] (tổng còn 3)
  giây 6: [1, 1, 1] rót 2 cốc -> [1, 0, 0] (tổng còn 1)
  giây 7: [1, 0, 0] rót 1 cốc -> [0, 0, 0] (tổng còn 0)
kết quả: 7
amount=[1, 4, 2]
  giây 1: [4, 2, 1] rót 2 cốc -> [3, 1, 1] (tổng còn 5)
  giây 2: [3, 1, 1] rót 2 cốc -> [2, 1, 0] (tổng còn 3)
  giây 3: [2, 1, 0] rót 2 cốc -> [1, 0, 0] (tổng còn 1)
  giây 4: [1, 0, 0] rót 1 cốc -> [0, 0, 0] (tổng còn 0)
kết quả: 4


In [13]:
def trace_formula(amount):
    s, m = sum(amount), max(amount)
    c = (s + 1) // 2
    print(f"amount={amount}: sum={s}, max={m}, ceil(sum/2)={c} -> đáp án = max({m}, {c}) = {max(m, c)}")
    return max(m, c)

for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0]):
    trace_formula(amt)

amount=[1, 4, 2]: sum=7, max=4, ceil(sum/2)=4 -> đáp án = max(4, 4) = 4
amount=[5, 4, 4]: sum=13, max=5, ceil(sum/2)=7 -> đáp án = max(5, 7) = 7
amount=[5, 0, 0]: sum=5, max=5, ceil(sum/2)=3 -> đáp án = max(5, 3) = 5


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def fillCups(self, amount: List[int]) -> int:
        return max(max(amount), (sum(amount) + 1) // 2)

print(Solution().fillCups([1, 4, 2]))
print(Solution().fillCups([5, 4, 4]))
print(Solution().fillCups([5, 0, 0]))

4
7
5


## 10. Vì sao thuật toán đúng

**Cận dưới.** Mỗi giây mỗi loại giảm tối đa 1 cốc, nên loại lớn nhất cần `>= max` giây. Mỗi giây tổng giảm tối đa 2 cốc, nên cần `>= ceil(sum/2)` giây. Vậy đáp án `>= max(max, ceil(sum/2))`.

**Đạt cận (bằng quy nạp theo chiến lược tham lam).** Gọi `T = max(max(amount), ceil(sum/2))`. Ta cho thấy luôn rót xong trong `T` giây.

- **Nếu `max >= ceil(sum/2)` (loại lớn nhất chiếm ít nhất nửa):** mỗi giây ghép loại lớn nhất với một cốc bất kỳ loại khác còn lại (nếu còn), nếu hết thì rót một mình. Loại lớn nhất giảm đúng 1 mỗi giây nên cần đúng `max` giây; trong thời gian đó mọi cốc khác đều kịp rót (vì tổng cốc khác `= sum - max <= max`). `T = max`.
- **Nếu `max < ceil(sum/2)`:** mỗi giây rót hai loại lớn nhất. Điều này **không** làm `max` vượt quá `ceil(còn_lại/2)` (vì ta giảm loại lớn nhất mỗi giây), nên tình huống "một loại chiếm hơn nửa" không bao giờ xuất hiện và mỗi giây luôn rót được đúng 2 cốc, ngoại trừ giây cuối nếu tổng lẻ. Số giây là `ceil(sum/2)`.

Hai cận dưới trùng cận trên nên đáp án là `max(max, ceil(sum/2))`.

## 11. Phân tích Time / Space Complexity

Gọi `S = sum(amount) <= 300`, `m = max(amount) <= 100`.

| Cách | Time | Space |
|---|---|---|
| 1. DP / đệ quy nhớ | `O(a*b*c)` trạng thái | `O(a*b*c)` |
| 2. Mô phỏng + sắp xếp | `O(S)` (tối đa ~300 vòng) | `O(1)` |
| 3. Mô phỏng + heap | `O(S)` (heap 3 phần tử) | `O(1)` |
| 4. Công thức `max(max, ceil(sum/2))` | `O(1)` | `O(1)` |
| 5. Sắp xếp + hai trường hợp | `O(1)` | `O(1)` |

Mảng chỉ có 3 phần tử nên mọi thứ "sắp xếp" là hằng số. Công thức `O(1)` là tối ưu về bậc; DP chỉ để đối chiếu và đòi hỏi `sys.setrecursionlimit` nếu giá trị lớn (ta chỉ test nhỏ).

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed` (DP chỉ chạy với giá trị nhỏ).

In [15]:
import random

fast = [SolutionSortLoop(), SolutionHeap(), SolutionFormula(), SolutionCases(), Solution()]
all_sol = [SolutionDP()] + fast

fixed = [
    ([1, 4, 2], 4), ([5, 4, 4], 7), ([5, 0, 0], 5),
    ([0, 0, 0], 0),            # không có cốc nào
    ([1, 1, 1], 2),            # 3 cốc: 2 giây
    ([2, 2, 2], 3),            # 6 cốc: 3 giây
    ([0, 3, 3], 3),            # hai loại cân bằng
    ([0, 1, 9], 9),            # một loại áp đảo: bị giới hạn bởi max
    ([100, 100, 100], 150),    # 300 cốc
    ([100, 0, 0], 100),
]
for amount, expected in fixed[:8]:
    for s in all_sol:
        assert s.fillCups(list(amount)) == expected, (type(s).__name__, amount, expected)
for amount, expected in fixed[8:]:
    for s in fast:
        assert s.fillCups(list(amount)) == expected, (type(s).__name__, amount, expected)
print("đã qua", len(fixed), "test cố định")

đã qua 10 test cố định


In [16]:
random.seed(2335)
for _ in range(400):                              # nhỏ: đối chiếu cả DP
    amount = [random.randint(0, 8) for _ in range(3)]
    ref = SolutionDP().fillCups(amount)
    for s in fast:
        assert s.fillCups(list(amount)) == ref, (type(s).__name__, amount)
for _ in range(2000):                             # lớn: đối chiếu mô phỏng với công thức
    amount = [random.randint(0, 100) for _ in range(3)]
    ref = SolutionSortLoop().fillCups(list(amount))
    for s in fast[1:]:
        assert s.fillCups(list(amount)) == ref, (type(s).__name__, amount)
print("400 test nhỏ (có DP) + 2000 test lớn đối chiếu: OK")

400 test nhỏ (có DP) + 2000 test lớn đối chiếu: OK


In [17]:
# thứ tự các phần tử không ảnh hưởng, và không sửa mảng đầu vào
import itertools
a = [3, 7, 2]
for p in itertools.permutations(a):
    assert Solution().fillCups(list(p)) == Solution().fillCups(a)
copy = list(a)
for s in fast:
    s.fillCups(a)
assert a == copy
print("hoán vị và không sửa input: OK")

hoán vị và không sửa input: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ dùng `ceil(sum/2)`, quên cận `max`

Với `[5,0,0]` tổng 5 -> `ceil(2.5) = 3` nhưng thực tế phải `5` giây vì chỉ có một loại, mỗi giây rót một cốc.

In [18]:
def sai_chi_tong(amount):
    return (sum(amount) + 1) // 2             # SAI: bỏ cận max

for amt in ([5, 0, 0], [1, 4, 2], [0, 1, 9]):
    print(amt, "sai:", sai_chi_tong(amt), "| đúng:", Solution().fillCups(amt))

[5, 0, 0] sai: 3 | đúng: 5
[1, 4, 2] sai: 4 | đúng: 4
[0, 1, 9] sai: 5 | đúng: 9


### Lỗi 2: Chia nguyên làm tròn xuống

`sum // 2` khi `sum` lẻ cho kết quả thiếu 1 giây. Ví dụ `[1,1,1]`: tổng 3, `3 // 2 = 1` nhưng cần 2.

In [19]:
def sai_chia_san(amount):
    return max(max(amount), sum(amount) // 2)     # SAI: thiếu làm tròn lên

for amt in ([1, 1, 1], [5, 4, 4], [1, 4, 2]):
    print(amt, "sai:", sai_chia_san(amt), "| đúng:", Solution().fillCups(amt))

[1, 1, 1] sai: 1 | đúng: 2
[5, 4, 4] sai: 6 | đúng: 7
[1, 4, 2] sai: 4 | đúng: 4


### Lỗi 3: Rót hai loại bất kỳ thay vì hai loại lớn nhất

Nếu ta cứ rót loại 0 và loại 1 mỗi giây thì loại 2 có thể còn rất nhiều ở cuối, lúc đó chỉ rót được 1 cốc/giây và tốn thêm giây.

In [20]:
def sai_tham_lam_xau(amount):
    a = list(amount)
    seconds = 0
    while sum(a) > 0:
        if a[0] > 0 and a[1] > 0:            # SAI: luôn ưu tiên loại 0, 1
            a[0] -= 1; a[1] -= 1
        elif a[0] > 0 and a[2] > 0:
            a[0] -= 1; a[2] -= 1
        elif a[1] > 0 and a[2] > 0:
            a[1] -= 1; a[2] -= 1
        else:
            for i in range(3):
                if a[i] > 0:
                    a[i] -= 1
                    break
        seconds += 1
    return seconds

for amt in ([1, 1, 4], [2, 2, 6], [1, 4, 2]):
    print(amt, "sai:", sai_tham_lam_xau(amt), "| đúng:", Solution().fillCups(amt))

[1, 1, 4] sai: 5 | đúng: 4
[2, 2, 6] sai: 8 | đúng: 6
[1, 4, 2] sai: 4 | đúng: 4


### Lỗi 4: Nhầm "một giây rót tối đa 2 cốc **khác loại**" thành "2 cốc bất kỳ"

Nếu cho phép rót 2 cốc cùng loại thì đáp án chỉ là `ceil(sum/2)`. Ràng buộc "khác loại" chính là lý do cận `max` xuất hiện.

In [21]:
amount = [5, 0, 0]
print("nếu được rót 2 cốc cùng loại:", (sum(amount) + 1) // 2)
print("theo đề (khác loại)         :", Solution().fillCups(amount))

nếu được rót 2 cốc cùng loại: 3
theo đề (khác loại)         : 5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "mỗi bước làm tối đa X, tìm số bước tối thiểu" -> nghĩ ngay **cận dưới từ từng ràng buộc** rồi lấy `max`.
- Nhiều "loại" cần xử lý song song, mỗi bước chọn các loại **lớn nhất** -> **tham lam + heap**.
- Chỉ 3 phần tử -> công thức `O(1)`; nhưng nếu nhiều hơn, dùng heap hoặc công thức `max(max, ceil(sum/2))` vẫn đúng (khi ghép tối đa 2 loại mỗi lần).
- Làm tròn lên: `(x + 1) // 2` hoặc `-(-x // 2)`.
- Bài cùng pattern: Task Scheduler (LeetCode 621), Reorganize String, Last Stone Weight.

## 15. Biến thể của bài

- **Số loại nước `k` bất kỳ** (không chỉ 3): đáp án vẫn `max(max, ceil(sum/2))`.
- **Rót tối đa 3 cốc khác loại mỗi giây:** cận dưới là `max(max, ceil(sum/3))`.
- **Trả về lịch rót cụ thể** (loại nào ở giây nào) bằng heap.
- **Có thời gian làm nguội** giữa hai lần rót cùng loại (Task Scheduler với cooldown).
- **Khác thời gian rót** từng loại (trọng số) — trở thành bài lập lịch phức tạp hơn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP / đệ quy nhớ | thử mọi cách | `O(abc)` | `O(abc)` | chỉ để kiểm chứng đáp án nhỏ |
| 2. Mô phỏng + sort | rót hai loại lớn nhất | `O(S)` | `O(1)` | dễ nghĩ, chưa thấy công thức |
| 3. Mô phỏng + heap | như 2 nhưng bằng heap | `O(S)` | `O(1)` | nhiều loại, cần lịch rót |
| 4. Công thức | `max(max, ceil(sum/2))` | `O(1)` | `O(1)` | nộp bài, tối ưu |
| 5. Hai trường hợp | `z >= x+y` hay không | `O(1)` | `O(1)` | giải thích vì sao đúng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().fillCups([2, 2, 2])` và `Solution().fillCups([7, 1, 1])`.

**Bài 2 (tính tay).** Với `[3, 6, 2]`, tính hai cận dưới `max` và `ceil(sum/2)` rồi đáp án.

**Bài 3 (điền TODO).** Hoàn thành `ceil_div(a, b)` trả về `ceil(a / b)` chỉ bằng số nguyên (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `fill_cups_k(amount)` cho **danh sách `amount` có số loại bất kỳ** (mỗi giây tối đa 2 loại khác nhau).

**Bài 5 (mini challenge).** Viết `schedule(amount)` trả về **lịch rót** dạng danh sách các cặp/loại mỗi giây (chỉ số loại) bằng heap; kiểm tra số giây bằng đáp án.

In [22]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def ceil_div_todo(a, b):
    # return (a + ?) // b
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2,2,2]` tổng 6 -> 3; `[7,1,1]` có `max = 7 >= 1 + 1` nên đáp án 7. Bài 2: `max = 6`, `sum = 11 -> 6`, đáp án 6.
- **Mức 2 (vừa):** Bài 3: `(a + b - 1) // b` với `a, b` nguyên dương. Bài 4: giữ công thức `max(max(a), ceil_div(sum(a), 2))`.
- **Mức 3 (sâu):** Bài 5: heap lưu `(-số_cốc_còn, loại)`; mỗi giây pop hai phần tử, ghi `(loại1, loại2)`, giảm 1 rồi push lại nếu còn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
import heapq

# Bài 1 và 2
assert Solution().fillCups([2, 2, 2]) == 3
assert Solution().fillCups([7, 1, 1]) == 7
assert max([3, 6, 2]) == 6 and (sum([3, 6, 2]) + 1) // 2 == 6 and Solution().fillCups([3, 6, 2]) == 6

# Bài 3
def ceil_div(a, b):
    return (a + b - 1) // b

# Bài 4
def fill_cups_k(amount):
    return max(max(amount), ceil_div(sum(amount), 2))

# Bài 5
def schedule(amount):
    heap = [(-x, i) for i, x in enumerate(amount) if x > 0]
    heapq.heapify(heap)
    plan = []
    while heap:
        n1, i1 = heapq.heappop(heap)
        step = [i1]
        back = [(n1 + 1, i1)]
        if heap:
            n2, i2 = heapq.heappop(heap)
            step.append(i2)
            back.append((n2 + 1, i2))
        for n, i in back:
            if n < 0:
                heapq.heappush(heap, (n, i))
        plan.append(tuple(step))
    return plan

assert ceil_div(7, 2) == 4 and ceil_div(6, 2) == 3 and ceil_div(0, 5) == 0
assert fill_cups_k([1, 4, 2]) == 4 and fill_cups_k([1, 1, 1, 1]) == 2 and fill_cups_k([9, 1, 1, 1, 1]) == 9
for amt in ([1, 4, 2], [5, 4, 4], [5, 0, 0], [3, 3, 3]):
    plan = schedule(amt)
    assert len(plan) == Solution().fillCups(amt)
    used = [0] * 3
    for step in plan:
        assert len(set(step)) == len(step)           # không rót hai cốc cùng loại trong một giây
        for i in step:
            used[i] += 1
    assert used == amt
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai cận dưới, lấy lớn hơn)

```python
class Solution:
    def fillCups(self, amount: List[int]) -> int:
        return max(max(amount), (sum(amount) + 1) // 2)
```

**Checklist**

- [ ] Cận 1: loại lớn nhất cần `max(amount)` giây (mỗi giây một cốc mỗi loại)?
- [ ] Cận 2: `ceil(sum / 2)` giây (mỗi giây tối đa 2 cốc)?
- [ ] Lấy `max` của hai cận, không chỉ một cái?
- [ ] Làm tròn **lên** (`(x + 1) // 2`), không phải `x // 2`?
- [ ] Nếu mô phỏng: luôn rót hai loại **nhiều nhất**?
- [ ] Đã thử `[0,0,0]`, `[5,0,0]`, `[1,1,1]`, `[100,100,100]`?
- [ ] Signature đúng `fillCups(self, amount: List[int]) -> int` và có `from typing import List`?